In [1]:
import sys
import requests

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from minio_config import configure_minio, minio_path

spark = (
    SparkSession.builder
    .appName("NYC Building Risk - Elasticsearch Indexing")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

configure_minio(spark)
spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/12 15:33:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/12 15:33:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/12 15:33:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/09/12 15:33:13 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/09/12 15:33:13 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


Spark: 3.4.0


In [2]:
ES_URL = "http://host.docker.internal:9200"

response = requests.get(
    ES_URL,
    timeout=10
)

print("Status:", response.status_code)
print("Elasticsearch version:", response.json()["version"]["number"])


Status: 200
Elasticsearch version: 8.15.3


In [3]:
# ==================================================
# LOAD SERVING BUILDING
# ==================================================

serving_building = spark.read.parquet(
    minio_path("serving/building")
)

print(
    "Serving building rows:",
    serving_building.count()
)

print(
    "Distinct building_id:",
    serving_building
    .select("building_id")
    .distinct()
    .count()
)

26/09/12 15:34:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Serving building rows: 197958


Distinct building_id: 197958


In [4]:
# ==================================================
# CREATE ELASTICSEARCH GEO_POINT
# ==================================================

serving_building_es = (
    serving_building

    .withColumn(
        "location",
        F.when(
            F.col("latitude").isNotNull()
            & F.col("longitude").isNotNull(),

            F.struct(
                F.col("latitude").alias("lat"),
                F.col("longitude").alias("lon")
            )
        )
    )
)

In [5]:
serving_building_es.select(
    "building_id",
    "search_address",
    "latitude",
    "longitude",
    "location"
).show(
    5,
    truncate=False
)

+-----------+-----------------+---------+----------+-----------------------+
|building_id|search_address   |latitude |longitude |location               |
+-----------+-----------------+---------+----------+-----------------------+
|BLD:1001873|83 WORTH STREET  |40.716594|-74.005184|{40.716594, -74.005184}|
|BLD:1004054|228 EAST BROADWAY|40.714359|-73.986707|{40.714359, -73.986707}|
|BLD:1004699|619 EAST 6 STREET|40.724036|-73.980518|{40.724036, -73.980518}|
|BLD:1005613|                 |null     |null      |null                   |
|BLD:1007822|159 6 AVENUE     |40.725581|-74.004012|{40.725581, -74.004012}|
+-----------+-----------------+---------+----------+-----------------------+
only showing top 5 rows



In [6]:
# ==================================================
# CHECK NULL / EMPTY BUILDING ADDRESSES
# ==================================================

missing_or_empty_address = (
    serving_building
    .filter(
        F.col("search_address").isNull()
        |
        (F.length(F.trim(F.col("search_address"))) == 0)
    )
)

print(
    "Missing or empty search_address:",
    missing_or_empty_address.count()
)

missing_or_empty_address.select(
    "building_id",
    "bin",
    "current_address",
    "property_address",
    "address_aliases",
    "property_id",
    "latitude",
    "longitude"
).show(
    20,
    truncate=False
)

Missing or empty search_address: 84
+-----------+-------+---------------+----------------+---------------+-----------+--------+---------+
|building_id|bin    |current_address|property_address|address_aliases|property_id|latitude|longitude|
+-----------+-------+---------------+----------------+---------------+-----------+--------+---------+
|BLD:1005613|1005613|               |null            |[]             |null       |null    |null     |
|BLD:1043029|1043029|               |null            |[]             |null       |null    |null     |
|BLD:1060005|1060005|               |null            |[]             |null       |null    |null     |
|BLD:1080100|1080100|               |null            |[]             |null       |null    |null     |
|BLD:1080948|1080948|               |null            |[]             |null       |null    |null     |
|BLD:1082752|1082752|               |null            |[]             |null       |null    |null     |
|BLD:1089075|1089075|               |null     

In [7]:
# ==================================================
# CLEAN SERVING BUILDING FOR ELASTICSEARCH
# ==================================================

serving_building_es = (
    serving_building_es

    .withColumn(
        "search_address",
        F.when(
            F.length(F.trim(F.col("search_address"))) > 0,
            F.col("search_address")
        )
        .otherwise(F.lit(None).cast("string"))
    )
)

print(
    "NULL search_address after cleanup:",
    serving_building_es
    .filter(F.col("search_address").isNull())
    .count()
)

print(
    "NULL location:",
    serving_building_es
    .filter(F.col("location").isNull())
    .count()
)

NULL search_address after cleanup: 84
NULL location: 183


In [8]:
# ==================================================
# UPDATE INDEX MAPPING
# ==================================================

mapping_update = {
    "properties": {
        "property_address": {
            "type": "text",
            "fields": {
                "keyword": {
                    "type": "keyword",
                    "ignore_above": 256
                }
            }
        }
    }
}

response = requests.put(
    f"{ES_URL}/building_risk_index/_mapping",
    json=mapping_update,
    timeout=30
)

print("Status:", response.status_code)
print(response.json())

Status: 200
{'acknowledged': True}


In [9]:
import json
import math
from datetime import date, datetime
from decimal import Decimal


def clean_for_json(value):

    if value is None:
        return None

    if isinstance(value, float):
        if math.isnan(value) or math.isinf(value):
            return None
        return value

    if isinstance(value, (date, datetime)):
        return value.isoformat()

    if isinstance(value, Decimal):
        return float(value)

    if isinstance(value, dict):
        return {
            k: clean_for_json(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            clean_for_json(v)
            for v in value
        ]

    return value

In [10]:
# ==================================================
# TEST BULK LOAD - 5 BUILDINGS
# ==================================================

test_rows = serving_building_es.limit(5).collect()

bulk_lines = []

for row in test_rows:

    document = clean_for_json(
        row.asDict(recursive=True)
    )

    building_id = document["building_id"]

    bulk_lines.append(
        json.dumps({
            "index": {
                "_index": "building_risk_index",
                "_id": building_id
            }
        })
    )

    bulk_lines.append(
        json.dumps(document)
    )


bulk_payload = "\n".join(bulk_lines) + "\n"


response = requests.post(
    f"{ES_URL}/_bulk?refresh=true",
    data=bulk_payload,
    headers={
        "Content-Type": "application/x-ndjson"
    },
    timeout=60
)

result = response.json()

print("HTTP Status:", response.status_code)
print("Bulk errors:", result.get("errors"))
print("Items:", len(result.get("items", [])))

26/09/12 15:46:07 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


HTTP Status: 200
Bulk errors: False
Items: 5


In [11]:
response = requests.get(
    f"{ES_URL}/building_risk_index/_count",
    timeout=30
)

print(response.json())

{'count': 5, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}}


In [12]:
# ==================================================
# FULL BULK LOAD - BUILDING RISK INDEX
# ==================================================

INDEX_NAME = "building_risk_index"
BATCH_SIZE = 1000

session = requests.Session()

total_sent = 0
batch_number = 0


def send_bulk_batch(documents):

    bulk_lines = []

    for document in documents:

        building_id = document["building_id"]

        bulk_lines.append(
            json.dumps({
                "index": {
                    "_index": INDEX_NAME,
                    "_id": building_id
                }
            })
        )

        bulk_lines.append(
            json.dumps(document)
        )

    payload = "\n".join(bulk_lines) + "\n"

    response = session.post(
        f"{ES_URL}/_bulk",
        data=payload,
        headers={
            "Content-Type": "application/x-ndjson"
        },
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    if result.get("errors"):

        failures = []

        for item in result["items"]:
            action = item.get("index", {})

            if action.get("error"):
                failures.append(action)

        print("Bulk failures:", failures[:5])

        raise RuntimeError(
            f"Bulk indexing failed. Failed documents: {len(failures)}"
        )

    return len(result["items"])

In [13]:
# ==================================================
# STREAM SPARK ROWS TO ELASTICSEARCH
# ==================================================

batch = []

for row in serving_building_es.toLocalIterator():

    document = clean_for_json(
        row.asDict(recursive=True)
    )

    batch.append(document)

    if len(batch) >= BATCH_SIZE:

        sent = send_bulk_batch(batch)

        total_sent += sent
        batch_number += 1

        print(
            f"Batch {batch_number}: "
            f"{sent} documents | "
            f"Total: {total_sent}"
        )

        batch = []


# Send final partial batch
if batch:

    sent = send_bulk_batch(batch)

    total_sent += sent
    batch_number += 1

    print(
        f"Batch {batch_number}: "
        f"{sent} documents | "
        f"Total: {total_sent}"
    )


print("\nFINISHED")
print("Total documents sent:", total_sent)

Batch 1: 1000 documents | Total: 1000
Batch 2: 1000 documents | Total: 2000
Batch 3: 1000 documents | Total: 3000
Batch 4: 1000 documents | Total: 4000
Batch 5: 1000 documents | Total: 5000
Batch 6: 1000 documents | Total: 6000
Batch 7: 1000 documents | Total: 7000
Batch 8: 1000 documents | Total: 8000
Batch 9: 1000 documents | Total: 9000
Batch 10: 1000 documents | Total: 10000
Batch 11: 1000 documents | Total: 11000
Batch 12: 1000 documents | Total: 12000
Batch 13: 1000 documents | Total: 13000
Batch 14: 1000 documents | Total: 14000
Batch 15: 1000 documents | Total: 15000
Batch 16: 1000 documents | Total: 16000
Batch 17: 1000 documents | Total: 17000
Batch 18: 1000 documents | Total: 18000
Batch 19: 1000 documents | Total: 19000
Batch 20: 1000 documents | Total: 20000
Batch 21: 1000 documents | Total: 21000
Batch 22: 1000 documents | Total: 22000
Batch 23: 1000 documents | Total: 23000


Exception ignored in: <function _local_iterator_from_socket.<locals>.PyLocalIterable.__del__ at 0x731af0ba84c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pyspark/rdd.py", line 293, in __del__
    for _ in self._read_iter:
  File "/usr/local/lib/python3.10/dist-packages/pyspark/serializers.py", line 152, in load_stream
    yield self._read_with_length(stream)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/serializers.py", line 174, in _read_with_length
    return self.loads(obj)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/serializers.py", line 472, in loads
    return cloudpickle.loads(obj, encoding=encoding)
  File "/usr/local/lib/python3.10/dist-packages/pyspark/sql/types.py", line 2006, in _create_row_inbound_converter
    def _create_row_inbound_converter(dataType: DataType) -> Callable:
KeyboardInterrupt: 
Exception ignored in sys.unraisablehook: <built-in function unraisablehook>
Traceback (most recent call last):
  Fil

In [ ]:
# ==================================================
# REFRESH INDEX
# ==================================================

response = requests.post(
    f"{ES_URL}/{INDEX_NAME}/_refresh",
    timeout=30
)

print("Refresh status:", response.status_code)

In [ ]:
# ==================================================
# VALIDATE DOCUMENT COUNT
# ==================================================

response = requests.get(
    f"{ES_URL}/{INDEX_NAME}/_count",
    timeout=30
)

result = response.json()

print(
    "Elasticsearch documents:",
    result["count"]
)

In [14]:
curl.exe --max-time 5 http://localhost:8889


SyntaxError: invalid syntax (3340583143.py, line 1)

In [15]:
print(serving_building_es.count())

197958


In [16]:
# ==================================================
# OPTIMIZE INDEX FOR BULK LOADING
# ==================================================

response = requests.put(
    f"{ES_URL}/building_risk_index/_settings",
    json={
        "index": {
            "refresh_interval": "-1"
        }
    },
    timeout=30
)

print(response.status_code)
print(response.json())

200
{'acknowledged': True}


In [17]:
import time

INDEX_NAME = "building_risk_index"

BATCH_SIZE = 250
MAX_RETRIES = 5

session = requests.Session()


def send_bulk_batch(documents):

    bulk_lines = []

    for document in documents:

        building_id = document["building_id"]

        bulk_lines.append(
            json.dumps({
                "index": {
                    "_index": INDEX_NAME,
                    "_id": building_id
                }
            })
        )

        bulk_lines.append(
            json.dumps(document)
        )

    payload = "\n".join(bulk_lines) + "\n"

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = session.post(
                f"{ES_URL}/_bulk?filter_path=errors,items.*.index.status,items.*.index.error",
                data=payload,
                headers={
                    "Content-Type": "application/x-ndjson"
                },
                timeout=60
            )

            response.raise_for_status()

            result = response.json()

            if result.get("errors"):

                failures = []

                for item in result.get("items", []):
                    action = item.get("index", {})

                    if action.get("error"):
                        failures.append(action)

                raise RuntimeError(
                    f"Bulk document errors: {len(failures)}"
                )

            return len(documents)

        except Exception as e:

            print(
                f"Attempt {attempt}/{MAX_RETRIES} failed:",
                str(e)
            )

            if attempt == MAX_RETRIES:
                raise

            wait_seconds = attempt * 2

            print(
                f"Waiting {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

In [18]:
# ==================================================
# SAFE FULL BUILDING LOAD
# ==================================================

total_processed = 0
batch_number = 0
batch = []

for row in serving_building_es.toLocalIterator():

    document = clean_for_json(
        row.asDict(recursive=True)
    )

    batch.append(document)

    if len(batch) >= BATCH_SIZE:

        sent = send_bulk_batch(batch)

        total_processed += sent
        batch_number += 1

        # Print progress every 10 batches
        if batch_number % 10 == 0:
            print(
                f"Batch {batch_number} | "
                f"Processed this run: {total_processed}"
            )

        batch = []

        # Small pause to reduce load on Elasticsearch
        time.sleep(0.05)


# Final partial batch
if batch:

    sent = send_bulk_batch(batch)

    total_processed += sent
    batch_number += 1


print("\nFINISHED")
print(
    "Documents processed this run:",
    total_processed
)

Batch 10 | Processed this run: 2500
Batch 20 | Processed this run: 5000
Batch 30 | Processed this run: 7500
Batch 40 | Processed this run: 10000
Batch 50 | Processed this run: 12500
Batch 60 | Processed this run: 15000
Batch 70 | Processed this run: 17500
Batch 80 | Processed this run: 20000
Batch 90 | Processed this run: 22500
Batch 100 | Processed this run: 25000
Batch 110 | Processed this run: 27500
Batch 120 | Processed this run: 30000
Batch 130 | Processed this run: 32500
Batch 140 | Processed this run: 35000
Batch 150 | Processed this run: 37500
Batch 160 | Processed this run: 40000
Batch 170 | Processed this run: 42500
Batch 180 | Processed this run: 45000
Batch 190 | Processed this run: 47500
Batch 200 | Processed this run: 50000
Batch 210 | Processed this run: 52500
Batch 220 | Processed this run: 55000
Batch 230 | Processed this run: 57500
Batch 240 | Processed this run: 60000
Batch 250 | Processed this run: 62500
Batch 260 | Processed this run: 65000
Batch 270 | Processed th

In [19]:
# ==================================================
# LOAD SERVING PROPERTY
# ==================================================

serving_property = spark.read.parquet(
    minio_path("serving/property")
)

print(
    "Serving property rows:",
    serving_property.count()
)

print(
    "Distinct property_id:",
    serving_property
    .select("property_id")
    .distinct()
    .count()
)

Serving property rows: 858284
Distinct property_id: 858284


In [20]:
# ==================================================
# PREPARE PROPERTY DATA FOR ELASTICSEARCH
# ==================================================

serving_property_es = (
    serving_property

    .withColumn(
        "search_address",
        F.when(
            F.length(F.trim(F.col("search_address"))) > 0,
            F.col("search_address")
        )
        .otherwise(F.lit(None).cast("string"))
    )

    .withColumn(
        "location",
        F.when(
            F.col("latitude").isNotNull()
            & F.col("longitude").isNotNull(),

            F.struct(
                F.col("latitude").alias("lat"),
                F.col("longitude").alias("lon")
            )
        )
    )
)

In [21]:
print(
    "NULL / empty search_address:",
    serving_property_es
    .filter(F.col("search_address").isNull())
    .count()
)

print(
    "NULL location:",
    serving_property_es
    .filter(F.col("location").isNull())
    .count()
)

serving_property_es.select(
    "property_id",
    "bbl",
    "search_address",
    "property_risk_score",
    "property_risk_level",
    "location"
).show(
    5,
    truncate=False
)

NULL / empty search_address: 577
NULL location: 937
+---------------+----------+---------------+-------------------+-------------------+-------------------------+
|property_id    |bbl       |search_address |property_risk_score|property_risk_level|location                 |
+---------------+----------+---------------+-------------------+-------------------+-------------------------+
|PROP:1000010010|1000010010|140 CARDER ROAD|32.88              |MEDIUM             |{40.6887632, -74.0187179}|
|PROP:1000010100|1000010100|CARDER ROAD    |null               |NO_DATA            |{40.6927301, -74.0138617}|
|PROP:1000010111|1000010111|ANDES ROAD     |9.56               |LOW                |{40.6929217, -74.0176373}|
|PROP:1000010150|1000010150|COMFORT ROAD   |null               |NO_DATA            |{40.6876026, -74.0155517}|
|PROP:1000010201|1000010201|1 ELLIS ISLAND |17.9               |LOW                |{40.6981883, -74.0413288}|
+---------------+----------+---------------+----------------

In [22]:
# ==================================================
# TEST BULK LOAD - 5 PROPERTIES
# ==================================================

PROPERTY_INDEX = "property_risk_index"

test_rows = serving_property_es.limit(5).collect()

bulk_lines = []

for row in test_rows:

    document = clean_for_json(
        row.asDict(recursive=True)
    )

    property_id = document["property_id"]

    bulk_lines.append(
        json.dumps({
            "index": {
                "_index": PROPERTY_INDEX,
                "_id": property_id
            }
        })
    )

    bulk_lines.append(
        json.dumps(document)
    )


bulk_payload = "\n".join(bulk_lines) + "\n"


response = requests.post(
    f"{ES_URL}/_bulk",
    data=bulk_payload,
    headers={
        "Content-Type": "application/x-ndjson"
    },
    timeout=60
)

result = response.json()

print("HTTP Status:", response.status_code)
print("Bulk errors:", result.get("errors"))
print("Items:", len(result.get("items", [])))

HTTP Status: 200
Bulk errors: False
Items: 5


In [23]:
# ==================================================
# SAFE PROPERTY BULK LOADER
# ==================================================

import time

PROPERTY_INDEX_NAME = "property_risk_index"

PROPERTY_BATCH_SIZE = 250
MAX_RETRIES = 5

property_session = requests.Session()


def send_property_bulk_batch(documents):

    bulk_lines = []

    for document in documents:

        property_id = document["property_id"]

        bulk_lines.append(
            json.dumps({
                "index": {
                    "_index": PROPERTY_INDEX_NAME,
                    "_id": property_id
                }
            })
        )

        bulk_lines.append(
            json.dumps(document)
        )

    payload = "\n".join(bulk_lines) + "\n"

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = property_session.post(
                f"{ES_URL}/_bulk?filter_path=errors,items.*.index.status,items.*.index.error",
                data=payload,
                headers={
                    "Content-Type": "application/x-ndjson"
                },
                timeout=60
            )

            response.raise_for_status()

            result = response.json()

            if result.get("errors"):

                failures = []

                for item in result.get("items", []):
                    action = item.get("index", {})

                    if action.get("error"):
                        failures.append(action)

                print(
                    "Example failures:",
                    failures[:3]
                )

                raise RuntimeError(
                    f"Bulk document errors: {len(failures)}"
                )

            return len(documents)

        except Exception as e:

            print(
                f"Attempt {attempt}/{MAX_RETRIES} failed:",
                str(e)
            )

            if attempt == MAX_RETRIES:
                raise

            wait_seconds = attempt * 2

            print(
                f"Waiting {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

In [24]:
# ==================================================
# FULL PROPERTY LOAD
# ==================================================

total_processed = 0
batch_number = 0
batch = []


for row in serving_property_es.toLocalIterator():

    document = clean_for_json(
        row.asDict(recursive=True)
    )

    batch.append(document)

    if len(batch) >= PROPERTY_BATCH_SIZE:

        sent = send_property_bulk_batch(batch)

        total_processed += sent
        batch_number += 1

        # Print every 20 batches = every 5,000 properties
        if batch_number % 20 == 0:

            print(
                f"Batch {batch_number} | "
                f"Processed this run: {total_processed}"
            )

        batch = []

        # Small pause to avoid overloading Elasticsearch
        time.sleep(0.05)


# Final partial batch
if batch:

    sent = send_property_bulk_batch(batch)

    total_processed += sent
    batch_number += 1


print("\nFINISHED")
print(
    "Documents processed this run:",
    total_processed
)

Batch 20 | Processed this run: 5000
Batch 40 | Processed this run: 10000
Batch 60 | Processed this run: 15000
Batch 80 | Processed this run: 20000
Batch 100 | Processed this run: 25000
Batch 120 | Processed this run: 30000
Batch 140 | Processed this run: 35000
Batch 160 | Processed this run: 40000
Batch 180 | Processed this run: 45000
Batch 200 | Processed this run: 50000
Batch 220 | Processed this run: 55000
Batch 240 | Processed this run: 60000
Batch 260 | Processed this run: 65000
Batch 280 | Processed this run: 70000
Batch 300 | Processed this run: 75000
Batch 320 | Processed this run: 80000
Batch 340 | Processed this run: 85000
Batch 360 | Processed this run: 90000
Batch 380 | Processed this run: 95000
Batch 400 | Processed this run: 100000
Batch 420 | Processed this run: 105000
Batch 440 | Processed this run: 110000
Batch 460 | Processed this run: 115000
Batch 480 | Processed this run: 120000
Batch 500 | Processed this run: 125000
Batch 520 | Processed this run: 130000
Batch 540 